# Chapter 9 — Text Pattern Matching with Regular Expressions

Source: [Automate the Boring Stuff with Python, 3rd Ed. — Chapter 9](https://automatetheboringstuff.com/3e/chapter9.html)

**Setup note:** Uses the `re` module, part of the Python standard library — no install needed.

This notebook favors *reading* over *writing*: each section has a short explanation followed by real, runnable code. Read it, run it, tweak a value and re-run it — that's the fastest way to build reading fluency.

---


## 9.1 Compiling a pattern and finding matches

A regular expression (*regex*) is a mini-language for describing text patterns. `re.compile()` turns a pattern string into a reusable Regex object; `.search()` finds the first match (or `None`), `.findall()` returns every match as a list.


In [1]:
import re

phone_regex = re.compile(r'\d\d\d-\d\d\d-\d\d\d\d')
mo = phone_regex.search('My number is 415-555-1234.')
print(mo.group())

text = 'Call 415-555-1234 or 212-555-9999.'
print(phone_regex.findall(text))


415-555-1234
['415-555-1234', '212-555-9999']


## 9.2 Groups: pulling out parts of a match

Parentheses in a pattern create *groups*, letting you extract just the parts you care about with `.group(n)` (group 0 is the whole match). The `|` (pipe) matches any one of several alternatives.


In [2]:
import re

phone_regex = re.compile(r'(\d\d\d)-(\d\d\d-\d\d\d\d)')
mo = phone_regex.search('Call 415-555-1234.')
print(mo.group(0))
print(mo.group(1))  # area code
print(mo.group(2))  # rest of the number

hero_regex = re.compile(r'Batman|Tina Fey')
print(hero_regex.search('Tina Fey and Batman are best friends.').group())


415-555-1234
415
555-1234
Tina Fey


## 9.3 Quantifiers: * + ? {n,m}

`?` makes the preceding group optional (0 or 1). `*` means 0 or more. `+` means 1 or more. `{n,m}` means between n and m repetitions. These let one pattern match many different but structurally similar strings.


In [3]:
import re

bat_regex = re.compile(r'Bat(wo)?man')     # matches 'Batman' or 'Batwoman'
print(bat_regex.findall('Batman and Batwoman'))

phone_regex = re.compile(r'\d{3}-\d{3}-\d{4}')
print(phone_regex.findall('415-555-1234 and 212-555-9999'))


['', 'wo']
['415-555-1234', '212-555-9999']


## 9.4 Greedy vs non-greedy matching

By default, quantifiers are *greedy*: they match as much text as possible. Adding a `?` after a quantifier (`*?`, `+?`) makes it *non-greedy*, matching as little text as possible instead.


In [4]:
import re

greedy = re.compile(r'<.*>')
nongreedy = re.compile(r'<.*?>')
text = '<a href="x">link</a>'
print(greedy.search(text).group())
print(nongreedy.search(text).group())


<a href="x">link</a>
<a href="x">


## 9.5 sub(): search and replace

`.sub(replacement, text)` returns a new string with every match replaced. Use `\1`, `\2`, ... in the replacement to reuse captured groups.


In [5]:
import re

names_regex = re.compile(r'Agent \w+')
print(names_regex.sub('CENSORED', 'Agent Alice told Agent Carol the secret.'))

phone_regex = re.compile(r'(\d\d\d)-(\d\d\d-\d\d\d\d)')
print(phone_regex.sub(r'(\1) \2', 'Call 415-555-1234.'))


CENSORED told CENSORED the secret.
Call (415) 555-1234.


## 9.6 Humre: building regexes without raw regex syntax

Raw regex syntax (`\d{3}-\d{4}`) is compact but hard to read months later. The third-party `Humre` ("HUman-readable REgex") module lets you build the exact same pattern out of readable Python function calls: `exactly(3, DIGIT)` instead of `\d{3}`, `group(...)` instead of `(...)`, `one_or_more(...)` instead of `+`. The result is a normal regex string — `Humre` is just a builder, not a different regex engine — so you still hand it to `re.search()`/`re.match()` exactly as before. The book rewrites its own phone-number example with it to show the readability gain directly.

In [6]:
from humre import *
import re

# The same phone-number pattern from earlier in this chapter, built with Humre instead of raw syntax
phone_pattern = group(
    exactly(3, DIGIT), '-', exactly(4, DIGIT)
)
print('Humre-built pattern:', phone_pattern)

text = 'Call 415-555-1234 or 800-555-9999 for support.'
matches = re.findall(phone_pattern, text)
print('matches:', matches)

# A more elaborate one: optional area code in parentheses, then the rest.
# Note: since this pattern has multiple parenthesized groups, re.findall() returns a
# tuple of every group's capture per match (an ordinary re quirk, not a Humre one) --
# so we pull out group(0), the whole match, via finditer() instead for a clean read.
full_phone_pattern = group(
    optional(group('(', exactly(3, DIGIT), ')', optional(WHITESPACE))),
    exactly(3, DIGIT), '-', exactly(4, DIGIT)
)
print('full pattern:', full_phone_pattern)
for m in re.finditer(full_phone_pattern, 'Call (415) 555-1234 or 555-9999.'):
    print(m.group(0))


Humre-built pattern: (\d{3}-\d{4})
matches: ['555-1234', '555-9999']
full pattern: (((\d{3})\s?)?\d{3}-\d{4})
555-1234
555-9999
